# ① VI OCR (Surya) — Minh Mệnh Chính Yếu  ·  notebook 1 / 2

Vietnamese side only, in a **clean Surya env** (no paddle — they conflict over Pillow).
Runs **step 1 (split + render)** and **step 2 (VI OCR with Surya + spell/confusable fix)**.
Writes everything to `out/<VOL>/` on Drive; notebook ②  picks it up for Hán OCR + align.

Why Surya: newer multilingual OCR, recovers tone marks + the b/h ascender that
flattens on this 1970s reprint (`tbần→thần`, `Trằm→Trẫm`) **at the source**.

Run order per volume: ① (this) → ② . Change **`VOL`** in §3.

## 1. Dependencies (Surya stack — NO paddle)  →  then **Runtime ▸ Restart**

In [ ]:
# Surya 0.14.5 (in-process, no docker) + light step-1/2 deps. Paddle absent (Pillow clash).
# Surya 0.14.5 imports transformers.QuantizedCacheConfig, REMOVED in transformers 4.57.
# ONE resolve: installing surya alone drags transformers 4.57 (broken). Pin all together
# so pip picks the compatible 4.51.3 set in a single pass.
# Do NOT split into 2 pip calls and do NOT Ctrl-C mid-install -> leaves transformers 4.57.
!pip -q install PyMuPDF opencv-python-headless pandas openpyxl tqdm
!pip -q install underthesea                 # step 2 sentence segmentation
!pip install "surya-ocr==0.14.5" "transformers==4.51.3" "tokenizers==0.21.1" "pillow>=10.2.0,<11.0.0"
import importlib.metadata as _m
for _p in ("surya-ocr","transformers","tokenizers","pillow"): print(_p, _m.version(_p))
_tv = _m.version("transformers")
assert _tv == "4.51.3", f"transformers={_tv}, expected 4.51.3 (re-run cell, do NOT Ctrl-C)"
print("\n>>> transformers 4.51.3 OK  -> Runtime ▸ Restart runtime, then run VERIFY")

### Verify GPU + Surya import (after restart)

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
import importlib.metadata as m; print("surya-ocr:", m.version("surya-ocr"))  # expect 0.14.5
from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor
det = DetectionPredictor(); rec = RecognitionPredictor()   # 0.14.x: both no-arg
print("Surya 0.14 API OK (no-arg constructors)")


torch 2.11.0+cu128 | CUDA: True
surya-ocr: 0.14.5


Surya 0.14 API OK (no-arg constructors)


## 2. Mount Drive + project

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
PROJECT = '/content/drive/MyDrive/minh_menh'   # CODE + inputs: pipeline/, data/<VOL>.pdf, assets/
MMCY    = '/content/mmcy'                       # WORK ROOT on LOCAL SSD (Drive FUSE corrupts many-file writes)
os.environ['MMCY_ROOT'] = MMCY                  # -> config DATA_DIR/OUT_DIR/ASSETS_DIR resolve LOCAL
# import pipeline code from Drive (read-only, safe); comment OFF the %cd line (magic eats it as path)
%cd $PROJECT
!mkdir -p {MMCY}/data {MMCY}/out
!rsync -a {PROJECT}/assets/ {MMCY}/assets/      # dicts/vocab -> local
print('code:', PROJECT, '| work root (LOCAL):', MMCY)
!ls data

Mounted at /content/drive
/content/drive/MyDrive/minh_menh
code: /content/drive/MyDrive/minh_menh | work root (LOCAL): /content/mmcy
vol1.pdf  vol2.pdf  vol3.pdf  vol4.pdf	vol5.pdf  vol6.pdf


## 3. Pick the volume
**Change `VOL` for each volume.**

In [ ]:
VOL = "vol1"          # <-- vol1 ... vol6
from pipeline import config
v = config.VIETNAMESE       # VI side re-OCRs the page images with Surya
print("VOL:", VOL, "| VI OCR: Surya")
print("fixers -> corrections + confusables + spell (always on) | min_token_len:",
      v["spell_min_token_len"])

## 4. Step 1 — split PDF + render Hán page images
Needs only PyMuPDF. Renders `out/<VOL>/pages_han/` that notebook ②  will OCR.

In [ ]:
!cp -f {PROJECT}/data/{VOL}.pdf {MMCY}/data/    # stage input PDF local (one read off Drive)
!python -m pipeline.step1_split_pdf --vol {VOL}  # renders pages to LOCAL out/, no Drive writes

step1 classify pages: 100% 638/638 [00:01<00:00, 428.03it/s]
[17:51:38] step1 INFO: [vol1] VI body starts at p8 | dropped 4 front-matter page(s): [0, 2, 4, 6]
[17:51:38] step1 INFO: [vol1] 638 pages | Hán starts at p224 | VI=205 HAN=405 (blanks dropped)
[17:51:38] step1 INFO: wrote vol1_vi.pdf (205 pages)
[17:51:38] step1 INFO: wrote vol1_han.pdf (405 pages)
step1 render pages_han:  92% 373/405 [01:11<00:08,  3.68it/s]

## 5. Step 2 — Vietnamese OCR (Surya) + confusable/spell fix + segment
Progress bar per page. Surya downloads weights on first run. Writes `out/<VOL>/vi_sentences.jsonl`.

In [ ]:
# clear any stale output first: a failed/empty surya run must SURFACE,
# not be masked by a leftover embedded-era vi_sentences.jsonl (<b> tags).
!rm -f {MMCY}/out/{VOL}/vi_sentences.jsonl
!python -m pipeline.step2_vietnamese --vol {VOL}

## 6. VI OCR quality check
OOV rate = cleanliness signal (lower better). Compare to a previous run if kept.

In [ ]:
import statistics as st
from pipeline.common import read_jsonl
rows = list(read_jsonl(config.OUT_DIR / VOL / "vi_sentences.jsonl"))
oov  = [r["oov_rate"] for r in rows if r.get("n_tokens")]
if not rows or not oov:
    raise SystemExit(f"⚠️ step2 produced {len(rows)} rows / {len(oov)} with tokens — "
                     "OCR likely returned empty. Re-check the surya run log (chars>0?).")
bold = sum("<b>" in r["text"] for r in rows)
print(f"{len(rows)} VI sentences | mean OOV {st.mean(oov):.3f} | median {st.median(oov):.3f} "
      f"| OOV>0.3: {sum(o>0.3 for o in oov)} ({100*sum(o>0.3 for o in oov)/len(oov):.1f}%) "
      f"| <b> tags: {bold}")
assert bold == 0, "<b> tags present -> reading a STALE embedded file, not surya output (check MMCY_ROOT=/content/mmcy)"
changed = [r for r in rows if r["text"] != r["raw_text"]]
print(f"\nfixers changed {len(changed)} sentences:")
for r in changed[:8]:
    print("  raw:", r["raw_text"][:80]); print("  fix:", r["text"][:80])
print("\n✅ VI side done (surya, <b>=0). Now run the hand-off cell below.")

1817 VI sentences | mean OOV 0.043 | median 0.025 | OOV>0.3: 10 (0.6%) | <b> tags: 0

fixers changed 336 sentences:
  raw: Xuống đến dời sau, chinh-yếu nhà Đường trong những năm T. inh.
  fix: Xuống đến dời sau, chính yếu nhà Đường trong những năm T. inh.
  raw: Nay kinh vang y chi Đức Hoàng-Thượng muốn trau-giòi chính sự cho ngày càng thêm 
  fix: Nay kinh vang y chi Đức Hoàng-Thượng muốn trau-giòi chính sự cho ngày càng thêm 
  raw: Kinh vâng lấy lời này l Ngày tháng chín năm thứ mười chin, vâng thánh chỉ du rằn
  fix: Kinh vâng lấy lời này l Ngày tháng chín năm thứ mười chin, vâng thánh chỉ du rằn
  raw: bản tiến trình, váng thánh chỉ dụ rằng: « Trẫm xem trong bộ Chinh yếu, nhiều chỗ
  fix: bản tiến trình, váng thánh chỉ dụ rằng: « Trẫm xem trong bộ Chính yếu, nhiều chỗ
  raw: Kinh vâng lấy lời này la Ngày tháng mười, niên-hiệu Triệu-trị năm đầu mới đặt cá
  fix: Kinh vâng lấy lời này la Ngày tháng mười, niên-hiệu Thiệu Trị năm đầu mới đặt cá
  raw: Khi ấy các vị trong sử-quản thươn

## 7. Hand-off to notebook ② (one Drive write, FUSE-safe)


In [ ]:
# FINAL hand-off (sequential): one Drive write = FUSE-safe. Bundle everything
# NB2 needs -> pages_han + split_manifest (step3) + vi_sentences (step4) + vi_boxes (draw_boxes QA overlay).
# pages_vi bundled too (QA / human review of the VI side, like pages_han).
!mkdir -p {PROJECT}/out_zips
!cd {MMCY}/out && zip -qr {PROJECT}/out_zips/{VOL}.zip \
    {VOL}/pages_han {VOL}/pages_vi {VOL}/split_manifest.json {VOL}/vi_sentences.jsonl {VOL}/vi_boxes.jsonl
!ls -lh {PROJECT}/out_zips/{VOL}.zip
print('hand-off done: out_zips/' + VOL + '.zip  ->  now run notebook ② top to bottom.')
